In [2]:
import pandas as pd
import os
import csv

# ─────────────────────────────────────────────
# CONFIGURAÇÃO
# ─────────────────────────────────────────────

PASTA = r"C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas"

# Mapeamento: valor atual → valor controlado v08
MAPEAMENTO = {

    # acao_institucional_smids
    "Defesa Civil fez triagem e acionou secretarias; Maria dos Anjos e Jesus Jose coordenaram resposta; familias nao precisaram sair dos imoveis; mutirao de limpeza com apoio da prefeitura": "acao_institucional_smids",
    "Sem limite de vagas - funciona como Banco de Talentos; destinado a maiores de 18 anos em vulnerabilidade social residentes ha 12 meses no municipio; links de inscricao disponibilizados pela Sec. de Governo": "acao_institucional_smids",
    "Organizado pelo Departamento das Mulheres; diretora Josefa Teixeira citada; empreendedoras do Jd. Amanda participaram como voluntarias; prefeito Zeze Gomes presente": "acao_institucional_smids",
    "Acao institucional SMIDS": "acao_institucional_smids",
    "Ampliacao de CRAS relevante para modelagem de cobertura territorial": "acao_institucional_smids",
    "Evidencia de capacidade institucional instalada; suporte empirico para interpretacao IVS alto mais IPST baixo; gestao funcional pode estar absorvendo pressao territorial; primeiro selo conquistado pelo municipio; secretaria Maria dos Anjos citada": "acao_institucional_smids",
    "Acao do Fundo Social; parceria Senai/Senac; programa gratuito com material incluido; depoimento formanda reforca importancia para mulheres": "acao_institucional_smids",
    "Prefeito Zeze visitou o local - foco em geracao de renda para mulheres": "acao_institucional_smids",

    # interface_ch
    "Nucleo monitora 1200 alvos 24h/dia; apresentacao contou com secretario de Seguranca Osvaldo Nico Goncalves e delegado-geral Artur Dian; impacto potencial em CH_03 por exposicao de menores a ambientes de risco digital": "interface_ch",
    "Evento relacionado a risco juvenil e possível vulnerabilidade social indireta": "interface_ch",
    "Amplia capacidade de resposta Capital Humano": "interface_ch",
    "Ação da Educação; interface com CH_03 e CH_07; presença do prefeito Zezé": "interface_ch",
    "Violencia de genero em espaco publico; interface PAEFI e CRAM": "interface_ch",
    "Interface direta com CH_05; benchmark regional": "interface_ch",
    "Impacto em PcD; interface com DPP PcD": "interface_ch",
    "Mortalidade juvenil em loteamento da RP_2; interface direta com CH_01; ausencia de equipamento de seguranca": "interface_ch",
    "Interface com CRAM e variaveis CH_04 e CH_05; arma com registro de furto apreendida; loteamento da RP_2": "interface_ch",
    "Criancas em situacao de risco dentro de equipamento de protecao social; interface CH_05 e vulnerabilidade infantil multipla; Jardim Conceicao RP_5; abrigo como loteamento de referencia": "interface_ch",
    "CH direto adolescente em conflito com a lei CH_08; RT contextual pressao socioeconomica indireta; Jardim Conceicao RP_5": "interface_ch",
    "Evidencia de capacidade institucional na educacao; interface direta com CH_07 e CH_03": "interface_ch",
    "Ativa CREAS e Conselho Tutelar; violencia intrafamiliar contra criancas/adolescentes; caso em acompanhamento por orgaos de protecao": "interface_ch",
    "Evento central e o roubo com envolvimento de menor de idade; desfecho positivo da acao policial nao apaga manifestacao de vulnerabilidade juvenil": "interface_ch",
    "Evidencia de violacao de direitos em familia monoparental - vulnerabilidade CH_05": "interface_ch",
    "Evento 12/04 das 9h as 12h - acessibilidade sensorial e orientacao parental": "interface_ch",
    "Indicador de violência doméstica e fragilidade familiar com dados auditáveis (TJ-SP)": "interface_ch",
    "Interface com CH_05; Conselho Tutelar acionado": "interface_ch",
    "Interface com CRAM e CH_04/CH_05; arma com registro de furto": "interface_ch",
    "Mortalidade juvenil RP_2; interface direta com CH_01": "interface_ch",

    # interface_rt
    "Ação institucional de promoção de empregabilidade; impacto potencial em inserção produtiva": "interface_rt",
    "Evidência empírica de inserção produtiva formal; dado pontual com potencial de validação estrutural": "interface_rt",
    "Contexto de custo de vida e pressao sobre renda": "interface_rt",
    "Contexto custo de vida e pressao sobre renda": "interface_rt",
    "Pressao de custo de vida sobre renda familiar RT_01": "interface_rt",
    "Equipamento voltado a idosos; interface direta com RT_04": "interface_rt",
    "Equipamento de saude voltado a populacao idosa; interface com RT_04": "interface_rt",
    "Sinal positivo de dinamica economica e trabalho formal": "interface_rt",
    "Alta continua e disseminada nas revendas; pressao direta sobre orcamento de familias de baixa renda; contexto: tensao EUA-Ira pressiona GLP importado": "interface_rt",
    "capacitacao_empreendedora": "interface_rt",
    "Impacto futuro em RT_04 Economia Prateada": "interface_rt",
    "Relevante para RT_06/IU_03 - mobilidade intermunicipal pendular Hortolandia": "interface_rt",

    # interface_iu
    "indicador direto de falha em saneamento; impacto mensuravel em IU_01": "interface_iu",
    "Conecta Av. Santana Jd. Amanda ao Centro e Nova Hortolandia; secretario de Obras Sergio Torrecillas cita rigor e agilidade cobrados da Rumo; prefeitura concluiu fresagem e sinalizacao no acesso": "interface_iu",
    "Indicador de expansao territorial impacto em IU_01 e IU_02": "interface_iu",
    "Indicador de expansao territorial; impacto em IU_01 e IU_02": "interface_iu",
    "Infraestrutura urbana em loteamento de interesse": "interface_iu",
    "Infraestrutura viaria com potencial de reorganizacao do fluxo urbano e valorizacao territorial; impacto indireto futuro sobre IU_01 e IU_02; RP_6": "interface_iu",
    "Impacto estrutural na reducao da vulnerabilidade habitacional; area em regularizacao fundiaria sem codbairro oficial": "interface_iu",
    "Obra estruturante de mobilidade urbana; impacto direto em IU_03; reduz barreira ferroviaria que fragmenta o municipio": "interface_iu",
    "Infraestrutura viaria reorganizacao fluxo urbano; RP_6": "interface_iu",
    "Engenheira ambiental Michelle Gouvea Martins confirmou alteracao pontual; nova analise laboratorial antes de retomada; populacao deve evitar coleta": "interface_iu",
    "sinal_administrativo_mobilidade": "interface_iu",
    "politica_habitacional_com_interface_social": "interface_iu",
    "Impacto estrutural na reducao da vulnerabilidade habitacional; area em regularizacao fundiaria": "interface_iu",

    # dado_auditavel
    "Dado auditavel corrobora IU_02": "dado_auditavel",
    "Rastreabilidade e evidencia tecnica como resposta ao contexto de auditoria": "dado_auditavel",
    "Rastreabilidade e evidencia tecnica": "dado_auditavel",
    "Dado administrativo oficial auditavel; volume mensal LMP permite construcao de serie temporal; base para monitoramento padrao CH_05 por periodo": "dado_auditavel",
    "Dado agregado mensal auditavel; volume e recorrencia permitem saida do evento isolado para padrao mensal; interface CH e RT; referencia para serie temporal do Atlas": "dado_auditavel",
    "Investimento dobrou em relacao a 2025 (R$ 152 mil) - inclui producao quilombola via Cooperquivale": "dado_auditavel",
    "Dado agregado mensal; referencia serie temporal do Atlas": "dado_auditavel",
    "Dado oficial auditavel; base para serie temporal CH_05": "dado_auditavel",
    "Nucleo mais presente no CadUnico com 10208 registros": "dado_auditavel",
    "Território com alta presença no CadUnico (10208 registros); evidência contextual de vulnerabilidade social": "dado_auditavel",
    "CIMH ativo desde 2022 no Remanso Campineiro - Rua Francisco Guimaraes de Oliveira 130": "dado_auditavel",

    # sinal_smids_ext
    "sinal_SMIDS_EXT": "sinal_smids_ext",
    "sinal_SMIDS_EXT; bairro_monte_sinai": "sinal_smids_ext",
    "Veiculo registrado em nome de terceiro; ocupantes recusaram atendimento do Samu e deixaram o local; caso registrado como dano ao patrimonio publico": "sinal_smids_ext",
    "Evento de violência urbana associado à pressão social no território": "sinal_smids_ext",
    "Vulnerabilidade individual com possivel isolamento social": "sinal_smids_ext",
    "Ocorrência de segurança no Jd. Amanda; bairro registrado em cod_loteamento": "sinal_smids_ext",
    "Homicidio em loteamento da RP_3; investigado pelo 2o DP; motivacao desconhecida; interface com seguranca publica e vulnerabilidade de renda": "sinal_smids_ext",
    "Homicidio em ambiente de trabalho informal; multidimensional CH mais RT indireto; ausencia de socorro institucional imediato; Jardim Amanda Quadra 1 RP_3": "sinal_smids_ext",
    "Porte ilegal de arma; localizacao confirmada": "sinal_smids_ext",
    "Vitima identificada; segundo evento negativo no Jd. Sao Bento; inicio de padrao territorial": "sinal_smids_ext",
    "Ocorrencia de seguranca no Jd. Amanda": "sinal_smids_ext",
    "Homicidio em loteamento RP_3; investigado pelo 2o DP": "sinal_smids_ext",

    # caso_ilustrativo
    "Parque Gabriel em área de vulnerabilidade; relevância territorial": "caso_ilustrativo",
    "Equipamento publico em loteamento de interesse": "caso_ilustrativo",
    "nome_jornalistico_remanso_das_aguas": "caso_ilustrativo",
    "Dado registrado em dim_programa_v05": "caso_ilustrativo",

    # impacto_latente
    "impacto_latente": "impacto_latente",

    # impacto_regional_hortolandia
    "Hortolandia Paulinia e Americana integram lista; todos pacientes recuperados sem internacao; sem obitos registrados em 2026": "impacto_regional_hortolandia",
    "Charge sugere que Monte Mor autorizou bloqueadores de ar em hidrometros para evitar medicao; sinal editorial de pressao social acumulada sobre ETE": "impacto_regional_hortolandia",
    "Articulacao federativa para regiao de crescimento populacional; complementa evento de 03/04": "impacto_regional_hortolandia",
    "Articulacao federativa para regiao de crescimento populacional": "impacto_regional_hortolandia",
    "Coordenador Renato Lopes Machado; secretaria-adjunta Jennifer Bazilio; SAMU ja atende chamadas de Sumare; adesao depende do Ministerio da Saude": "impacto_regional_hortolandia",
    "Relevante para pressao de saude regional - contexto IU/CH regional": "impacto_regional_hortolandia",

    # multiplos_loteamentos
    "multiplos_loteamentos": "multiplos_loteamentos",
    "Cruzar com loteamentos afetados": "multiplos_loteamentos",

    # fora_escopo_smids
    "Equipamento de saude fora do escopo SMIDS": "fora_escopo_smids",
    "Acao pontual nao entra no catalogo de programas": "fora_escopo_smids",
    "Publico-alvo: servidores municipais; fora do escopo IVS direto; sinaliza capacidade de gestao institucional da administracao": "fora_escopo_smids",
    "Local provisorio por obra do viaduto Vila Real (codbairro 63 RP_6); impacto cultural indireto; sem relacao IVS direta": "fora_escopo_smids",

    # contexto_sociopolitico
    "Contexto político relevante para período de apresentação em abril": "contexto_sociopolitico",
    "Contexto politico relevante para periodo de apresentacao em abril": "contexto_sociopolitico",
    "Evento de governanca regional sediado em Hortolandia; fortalece controle social e transparencia publica": "contexto_sociopolitico",
    "Camara de Hortolândia como realizadora; tema transparencia em setores publicos; fortalece ambiente institucional que sustenta o Atlas Social": "contexto_sociopolitico",
    "Instrumento estruturante que define a base territorial do Atlas — RPs e loteamentos; legitima politicamente o modelo; acompanhar desdobramentos para ajustes em dim_loteamento": "contexto_sociopolitico",
    "Instrumento que define base territorial do Atlas; RPs e loteamentos": "contexto_sociopolitico",
    "Reorganizacao interna administrativa; sem impacto direto nas variaveis IVS": "contexto_sociopolitico",

    # tarefa_pendente
    "Verificar nome oficial do programa com Sec. Meio Ambiente - pendente": "tarefa_pendente",
    "Verificar nome oficial do programa": "tarefa_pendente",
    "Indicacao legislativa - aguarda aprovacao monitorar": "tarefa_pendente",
    "Indicacao legislativa aguarda aprovacao": "tarefa_pendente",

    # duvida_metodologica
    "Deslocamento em rodovia as 5h sugere trabalhador em trajeto laboral precoce; motorista nao identificado; articulacao possivel com IU_03 se confirmado perfil de renda": "duvida_metodologica",

    # aproximacao_variavel
    "acao_preventiva_saude_com_dado_operacional": "aproximacao_variavel",
}

# ─────────────────────────────────────────────
# EXECUÇÃO
# ─────────────────────────────────────────────

nao_mapeados = []
arquivos_processados = 0
arquivos_sem_coluna = 0

for arquivo in sorted(os.listdir(PASTA)):
    if not arquivo.endswith(".csv"):
        continue
    if arquivo == "corpus_resumo_periodos_v07.csv":
        continue

    caminho = os.path.join(PASTA, arquivo)
    df = pd.read_csv(caminho, dtype=str, quoting=csv.QUOTE_ALL)

    if "observacao" not in df.columns:
        df["observacao"] = ""
        df["nota_analista"] = ""
        arquivos_sem_coluna += 1
    else:
        df["nota_analista"] = df["observacao"].fillna("")

        def mapear(valor):
            if pd.isna(valor) or str(valor).strip() == "":
                return ""
            valor_limpo = str(valor).strip()
            if valor_limpo in MAPEAMENTO:
                return MAPEAMENTO[valor_limpo]
            else:
                nao_mapeados.append({"arquivo": arquivo, "valor": valor_limpo})
                return valor_limpo

        df["observacao"] = df["observacao"].apply(mapear)

    colunas = [c for c in df.columns if c not in ["observacao", "nota_analista"]]
    colunas_final = colunas + ["observacao", "nota_analista"]
    df = df[colunas_final]

    df.to_csv(caminho, index=False, quoting=csv.QUOTE_ALL, encoding="utf-8")
    arquivos_processados += 1

# ─────────────────────────────────────────────
# RELATÓRIO FINAL
# ─────────────────────────────────────────────

print(f"Arquivos processados : {arquivos_processados}")
print(f"Arquivos sem coluna  : {arquivos_sem_coluna}")
print(f"Valores nao mapeados : {len(nao_mapeados)}")

if nao_mapeados:
    print()
    print("⚠️  VALORES QUE PRECISAM DE REVISÃO MANUAL:")
    for item in nao_mapeados:
        print(f"  [{item['arquivo']}]  {item['valor']}")
else:
    print()
    print("✅ Todos os valores foram mapeados com sucesso.")


Arquivos processados : 30
Arquivos sem coluna  : 0
Valores nao mapeados : 90

⚠️  VALORES QUE PRECISAM DE REVISÃO MANUAL:
  [2026_03_01_tribuna_liberal.csv]  acao_institucional_smids
  [2026_03_01_tribuna_liberal.csv]  acao_institucional_smids
  [2026_03_01_tribuna_liberal.csv]  acao_institucional_smids
  [2026_03_01_tribuna_liberal.csv]  acao_institucional_smids
  [2026_03_01_tribuna_liberal.csv]  acao_institucional_smids
  [2026_03_01_tribuna_liberal.csv]  interface_ch
  [2026_03_03_tribuna_liberal.csv]  interface_iu
  [2026_03_03_tribuna_liberal.csv]  impacto_regional_hortolandia
  [2026_03_03_tribuna_liberal.csv]  acao_institucional_smids
  [2026_03_03_tribuna_liberal.csv]  sinal_smids_ext
  [2026_03_03_tribuna_liberal.csv]  impacto_regional_hortolandia
  [2026_03_04_tribuna_liberal.csv]  interface_iu
  [2026_03_04_tribuna_liberal.csv]  sinal_smids_ext
  [2026_03_04_tribuna_liberal.csv]  sinal_smids_ext
  [2026_03_14_tribuna_liberal.csv]  interface_ch
  [2026_03_14_tribuna_liberal.